# 07C – LIME Local Explainability

Enterprise notebook for explaining individual bankruptcy predictions using **LIME**.

## Business Objective

Use LIME (Local Interpretable Model-agnostic Explanations) to explain why the model predicted bankruptcy risk for a single company and compare these insights with SHAP.

In [1]:
# Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer

In [2]:
MODEL_PATH='../models/production_bankruptcy_model.joblib'
DATA_PATH='../data/datasets/american_bankruptcy.csv'
INSTANCE_INDEX=0

model=joblib.load(MODEL_PATH)
df=pd.read_csv(DATA_PATH)

if 'status_label' in df.columns:
    y=df['status_label'].map({'alive':0,'failed':1})
    X=df.drop(columns=[c for c in ['status_label','company_name'] if c in df.columns])
elif 'target' in df.columns:
    y=df['target']
    X=df.drop(columns=[c for c in ['target','company_name'] if c in df.columns])
else:
    raise ValueError('Target column not found')

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)

instance=X_test.iloc[INSTANCE_INDEX]

In [3]:
explainer=LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['Healthy','Bankrupt'],
    mode='classification',
    discretize_continuous=True
)

exp=explainer.explain_instance(
    instance.values,
    model.predict_proba,
    num_features=15
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [4]:
# Save interactive explanation
exp.save_to_file('lime_explanation.html')

weights=pd.DataFrame(
    exp.as_list(),
    columns=['Feature','Contribution']
)

weights.to_csv('lime_feature_contributions.csv',index=False)
weights

,Feature,Contribution
0,X8 <= 34.88,0.013135
1,7.52 < X11 <= 248.80,-0.006593
2,1.18 < X3 <= 7.89,-0.006474
3,3.25 < X7 <= 22.72,-0.004737
4,185.60 < X9 <= 1039.80,-0.004180
5,63.09 < X13 <= 342.11,-0.004162
6,8.75 < X14 <= 43.02,-0.003232
7,37.28 < X10 <= 212.68,-0.003230
8,18.84 < X1 <= 100.00,0.003119
9,1.62 < X6 <= 39.94,-0.002438


## Business Interpretation

- Positive contributions increase the predicted bankruptcy probability.
- Negative contributions reduce the predicted bankruptcy probability.
- LIME provides an intuitive local explanation for one prediction, making it valuable for model validation and stakeholder communication.

## Deliverables

- `lime_explanation.html`
- `lime_feature_contributions.csv`

### Portfolio Value
This notebook demonstrates model-agnostic local explainability and complements SHAP-based explanations.